In [ ]:
import os
import shutil
import pandas as pd
from PIL import Image
from tqdm import tqdm



In [ ]:
# =========================
# CONFIG
# =========================

RAW_DATA_DIR = "data/RAW_DATA_PLACEHOLDER"  
ANNOTATIONS_CSV = os.path.join(RAW_DATA_DIR, "annotations.csv")
IMAGES_ROOT = RAW_DATA_DIR

OUTPUT_DIR = "data/IMAGEFOLDER_SPLIT"

IMAGE_SIZE = (X, X)
OUTPUT_FORMAT = "jpg"   
JPEG_QUALITY = 95



In [ ]:
# =========================
# LOAD ANNOTATIONS
# =========================

df = pd.read_csv(ANNOTATIONS_CSV)

required_cols = ["filepath", "split", "cell class"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

print("Dataset size:", len(df))
print(df["split"].value_counts())
print(df["cell class"].value_counts())

# =========================
# CREATE OUTPUT FOLDERS
# =========================

for split in ["train", "val", "test"]:
    split_df = df[df["split"] == split]
    for class_name in split_df["cell class"].unique():
        os.makedirs(os.path.join(OUTPUT_DIR, split, class_name), exist_ok=True)



In [ ]:
# =========================
# PROCESS AND COPY IMAGES
# =========================

missing_files = []
failed_files = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    rel_path = row["filepath"]          # e.g. images/X.tiff
    split = row["split"]                # train / val / test
    class_name = row["cell class"]      # basophil, lymphocyte, etc.

    src_path = os.path.join(IMAGES_ROOT, rel_path)

    if not os.path.exists(src_path):
        missing_files.append(src_path)
        continue

    try:
        img = Image.open(src_path).convert("RGB")
        img = img.resize(IMAGE_SIZE)

        original_name = os.path.splitext(os.path.basename(rel_path))[0]
        new_filename = f"{original_name}.{OUTPUT_FORMAT}"

        dst_path = os.path.join(
            OUTPUT_DIR,
            split,
            class_name,
            new_filename
        )

        if OUTPUT_FORMAT.lower() == "jpg":
            img.save(dst_path, "JPEG", quality=JPEG_QUALITY)
        elif OUTPUT_FORMAT.lower() == "png":
            img.save(dst_path, "PNG")
        else:
            raise ValueError("OUTPUT_FORMAT must be 'jpg' or 'png'")

    except Exception as e:
        failed_files.append((src_path, str(e)))



In [ ]:
# =========================
# SAVE CLEANED ANNOTATIONS
# =========================

cleaned_csv_path = os.path.join(OUTPUT_DIR, "annotations_final.csv")
df.to_csv(cleaned_csv_path, index=False)

# =========================
# REPORT
# =========================

print("\nPreprocessing complete.")
print("Final dataset saved to:", OUTPUT_DIR)
print("Missing files:", len(missing_files))
print("Failed files:", len(failed_files))

if missing_files:
    pd.DataFrame({"missing_files": missing_files}).to_csv(
        os.path.join(OUTPUT_DIR, "missing_files.csv"),
        index=False
    )

if failed_files:
    pd.DataFrame(failed_files, columns=["file", "error"]).to_csv(
        os.path.join(OUTPUT_DIR, "failed_files.csv"),
        index=False
    )



In [ ]:
# =========================
# CLASS COUNT REPORT
# =========================

class_counts = []

for split in ["train", "val", "test"]:
    split_path = os.path.join(OUTPUT_DIR, split)
    for class_name in sorted(os.listdir(split_path)):
        class_path = os.path.join(split_path, class_name)
        if os.path.isdir(class_path):
            count = len(os.listdir(class_path))
            class_counts.append({
                "split": split,
                "class": class_name,
                "count": count
            })

class_counts_df = pd.DataFrame(class_counts)
class_counts_df.to_csv(os.path.join(OUTPUT_DIR, "class_counts.csv"), index=False)

print("\nClass counts:")
print(class_counts_df)